In [2]:
import joblib
MODEL_PATH = 'language_detector.pkl'

In [ ]:
import unicodedata

def detect_script(text):
    for char in text:
        if '\u0600' <= char <= '\u06FF':
            return 'ar'   # Arabic script
        if '\u4e00' <= char <= '\u9fff':
            return 'zh'   # Chinese script
        if '\u0400' <= char <= '\u04FF':
            return 'ru'   # Cyrillic script
        if '\u0900' <= char <= '\u097F':
            return 'hi'   # Devanagari (Hindi)
    return None  # Latin script, can't determine from script alone

def detect_language(text, pipeline):
    # Step 1: Short Latin text — use a greeting dictionary
    if len(text.split()) < 3:
        # Step 2: Script-based detection for non-Latin scripts
        script_lang = detect_script(text)
        if script_lang:
            return script_lang


        greetings = {
            # English
            'hi': 'en', 'hello': 'en', 'hey': 'en', 'bye': 'en',
            'goodbye': 'en', 'thanks': 'en', 'thank': 'en', 'please': 'en',
            'yes': 'en', 'no': 'en', 'okay': 'en', 'ok': 'en',

            # French
            'bonjour': 'fr', 'bonsoir': 'fr', 'bonne': 'fr',
            'merci': 'fr', 'aurevoir': 'fr', 'oui': 'fr',
            'allô': 'fr', 'allo': 'fr',

            # Spanish
            'hola': 'es', 'buenos': 'es', 'buenas': 'es', 'adios': 'es',
            'adiós': 'es', 'gracias': 'es', 'sí': 'es', 'chao': 'es',

            # Italian
            'ciao': 'it', 'buongiorno': 'it', 'buonasera': 'it',
            'salve': 'it', 'arrivederci': 'it', 'grazie': 'it',
            'prego': 'it', 'sì': 'it',

            # German
            'hallo': 'de', 'guten': 'de', 'tschüss': 'de', 'tschuss': 'de',
            'danke': 'de', 'bitte': 'de', 'nein': 'de',
            'servus': 'de', 'moin': 'de',

            # Portuguese
            'olá': 'pt', 'ola': 'pt', 'oi': 'pt', 'tchau': 'pt',
            'obrigado': 'pt', 'obrigada': 'pt', 'sim': 'pt',

            # Dutch
            'hoi': 'nl', 'dag': 'nl', 'doei': 'nl',
            'dank': 'nl', 'nee': 'nl', 'goedemorgen': 'nl',

            # Swedish
            'hej': 'sv', 'tjenare': 'sv', 'hallå': 'sv', 'halla': 'sv',
            'tack': 'sv', 'adjö': 'sv', 'adjo': 'sv',

            # Turkish
            'merhaba': 'tr', 'selam': 'tr', 'günaydın': 'tr',
            'teşekkür': 'tr', 'evet': 'tr', 'hayır': 'tr', 'hoşça': 'tr',

            # Polish
            'cześć': 'pl', 'czesc': 'pl', 'dzień': 'pl', 'dzien': 'pl',
            'dobry': 'pl', 'dziękuję': 'pl', 'dziekuje': 'pl',
            'tak': 'pl', 'nie': 'pl',

            # Romanian
            'bună': 'ro', 'buna': 'ro', 'mulțumesc': 'ro',
            'multumesc': 'ro', 'noapte': 'ro',

            # Indonesian/Malay
            'halo': 'id', 'hai': 'id', 'selamat': 'id', 'terima': 'id',
            'tidak': 'id', 'sampai': 'id',

            # Ambiguous words assigned to their most common language
            'ja': 'de',       # used in DE, NL, SV — most common in German
            'da': 'ru',       # used in RO, SL — most common in Russian (Cyrillic handled separately)
            'si': 'es',       # used in ES, IT, FR — most common in Spanish
            'salut': 'fr',    # used in FR, RO — most common in French
            'nu': 'ro',       # used in RO, NL — most common in Romanian
            'non': 'fr',      # used in FR, IT — most common in French
            'ya': 'es',       # used in ID, ES, RU — most common in Spanish informal
        }

        first_word = text.lower().strip().split()[0] if text.strip() else ''
        if first_word in greetings:
            return greetings[first_word]

    # Step 3: Fall back to the ML model for everything else
    return pipeline.predict([text])[0]

In [5]:
# Verify load + inference
loaded_pipeline = joblib.load(MODEL_PATH)

test_sentences = [
    "صابحو",
    "ازيك",
    "Hi",
    "Hello",
    "Bonjour",
    "salut",
    "Hi, there",
    "Hello, there",
    "Hello, how are you feeling today?",
    "مرحبا، كيف حالك اليوم؟",
    "Bonjour, comment vous sentez-vous ?",
    "Hola, ¿cómo te sientes hoy?",
    "Wie geht es Ihnen heute?",
    "Привет, как вы себя чувствуете?",
    "今天你感觉怎么样？"
]

print("=== Language Detection Demo ===")
for text in test_sentences:
    pred = detect_language(text, loaded_pipeline)
    print(f"[{pred}] {text}")

=== Language Detection Demo ===
[ar] صابحو
[ar] ازيك
[en] Hi
[en] Hello
[fr] Bonjour
[fr] salut
[en] Hi, there
[en] Hello, there
[en] Hello, how are you feeling today?
[ar] مرحبا، كيف حالك اليوم؟
[fr] Bonjour, comment vous sentez-vous ?
[es] Hola, ¿cómo te sientes hoy?
[de] Wie geht es Ihnen heute?
[ru] Привет, как вы себя чувствуете?
[zh] 今天你感觉怎么样？
